In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

def load_data(file_path):
    sentences = []
    labels = []
    current_sentence = []
    current_labels = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split()
                if len(parts) >= 2:
                    current_sentence.append(parts[0])
                    current_labels.append(parts[1])
            else:
                if current_sentence:
                    sentences.append(current_sentence)
                    labels.append(current_labels)
                    current_sentence = []
                    current_labels = []
    if current_sentence:
        sentences.append(current_sentence)
        labels.append(current_labels)
    return sentences, labels

train_sentences, train_labels = load_data('train_corrected.txt')
test_sentences, test_labels = load_data('test_corrected.txt')

all_words = sorted(list(set([w for s in train_sentences + test_sentences for w in s])))
word2idx = {w: i + 1 for i, w in enumerate(all_words)}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = len(word2idx)

all_labels = sorted(list(set([l for sublist in train_labels + test_labels for l in sublist])))
label_encoder = LabelEncoder()
label_encoder.fit(all_labels)

def encode_sequences(sentences, labels, word2idx, label_encoder, max_len=100):
    X = []
    y = []
    for i in range(len(sentences)):
        sent = sentences[i]
        lab = labels[i]
        sent_idx = [word2idx.get(w, word2idx["<UNK>"]) for w in sent]
        lab_idx = label_encoder.transform(lab)
        if len(sent_idx) < max_len:
            sent_idx = sent_idx + [0] * (max_len - len(sent_idx))
            lab_idx = list(lab_idx) + [-100] * (max_len - len(lab_idx))
        else:
            sent_idx = sent_idx[:max_len]
            lab_idx = lab_idx[:max_len]
        X.append(sent_idx)
        y.append(lab_idx)
    return np.array(X), np.array(y)

MAX_LEN = 100
X_train, y_train = encode_sequences(train_sentences, train_labels, word2idx, label_encoder, MAX_LEN)
X_test, y_test = encode_sequences(test_sentences, test_labels, word2idx, label_encoder, MAX_LEN)

class NERDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = DataLoader(NERDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(NERDataset(X_test, y_test), batch_size=32, shuffle=False)

embedding_dim = 300
vocab_size = len(word2idx)
embedding_matrix = np.zeros((vocab_size, embedding_dim))

try:
    with open('cc.id.300.vec', encoding='utf-8') as f:
        next(f)
        print("File found successfully.")
        for line in f:
            values = line.rstrip().rsplit(' ')
            word = values[0]
            if word in word2idx:
                coefs = np.asarray(values[1:], dtype='float32')
                embedding_matrix[word2idx[word]] = coefs
except FileNotFoundError:
    print("File is nowhere to be found :((((((")
    pass

print("BiLSTM model is initialized")
class BiLSTM_NER(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, embeddings):
        super(BiLSTM_NER, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.from_numpy(embeddings))
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        x = self.dropout(x)
        x = self.fc(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTM_NER(vocab_size, embedding_dim, 256, len(all_labels), embedding_matrix).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Model is in training phase..")
for epoch in range(5):
    model.train()
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.view(-1, len(all_labels)), targets.view(-1))
        loss.backward()
        optimizer.step()

print("Model predicting..")
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=2).cpu().numpy()
        targets = targets.numpy()
        for i in range(len(targets)):
            valid_indices = targets[i] != -100
            all_targets.extend(label_encoder.inverse_transform(targets[i][valid_indices]))
            all_preds.extend(label_encoder.inverse_transform(preds[i][valid_indices]))

unique_labels = sorted(list(set(all_targets)))
print(f"Accuracy Score: {accuracy_score(all_targets, all_preds) * 100:.2f}%")
print(classification_report(all_targets, all_preds, digits=4, zero_division=0))

File found successfully.
BiLSTM model is initialized
Model is in training phase..
Model predicting..
Accuracy Score: 92.56%
              precision    recall  f1-score   support

       B-CRD     0.8086    0.9155    0.8587      1006
       B-DAT     0.9664    0.9540    0.9602       783
       B-EVT     0.7616    0.6319    0.6907       182
       B-FAC     0.4066    0.3978    0.4022        93
       B-GPE     0.8513    0.8387    0.8450      1358
       B-LAW     0.7273    0.4571    0.5614        35
       B-LOC     0.6494    0.5208    0.5780       288
       B-MON     0.9556    0.9331    0.9442       254
       B-NOR     0.7678    0.8002    0.7837       901
       B-ORD     0.8851    0.5580    0.6844       138
       B-ORG     0.7033    0.6307    0.6650       853
       B-PER     0.9204    0.7214    0.8089      1443
       B-PRC     0.9101    0.9149    0.9125       188
       B-PRD     0.6917    0.5092    0.5866       868
       B-QTY     0.8118    0.5130    0.6287       269
       B-RE